In [1]:
import os

import requests
from dotenv import load_dotenv
from openai import AsyncOpenAI
from pydantic import BaseModel, Field

from agents import (
    Agent,
    GuardrailFunctionOutput,
    InputGuardrailTripwireTriggered,
    OpenAIChatCompletionsModel,
    Runner,
    function_tool,
    input_guardrail,
    trace,
)

load_dotenv(override=True)


True

In [2]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"
PUSHOVER_MAX_LEN = 1024


def send_push(message: str) -> None:
    """Send a Pushover notification (guardrails and tools call this)."""
    if not pushover_user or not pushover_token:
        print("Pushover not configured; skipping notification.")
        return
    text = message if len(message) <= PUSHOVER_MAX_LEN else message[: PUSHOVER_MAX_LEN - 3] + "..."
    print(f"Push: {text[:200]}{'...' if len(text) > 200 else ''}")
    payload = {"user": pushover_user, "token": pushover_token, "message": text}
    try:
        requests.post(pushover_url, data=payload, timeout=10)
    except requests.RequestException as e:
        print(f"Pushover request failed: {e}")


@function_tool
def push(message: str):
    """Notify via Pushover (e.g. when the agent chooses to alert you)."""
    send_push(message)

In [3]:

OPENROUTER_BASE_URL="https://openrouter.ai/api/v1"
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
openrouter = AsyncOpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)


openrouter_client = OpenAIChatCompletionsModel(
    model="openai/gpt-4o-mini",
    openai_client=openrouter,
)

class QueryCheck(BaseModel):
    is_valid: bool = Field(description="True if the query is suitable for research.")
    reason: str = Field(description="Brief reason if not valid.")

query_check_agent = Agent(
    name="QueryCheck",
    instructions="Check if the user query is valid for sports research: not empty, at least a few words, and a real question.",
    output_type=QueryCheck,
    model=openrouter_client,
)

In [4]:
input_guardrail
async def guardrail_query(ctx, agent, message):
    result = await Runner.run(query_check_agent, message, context=ctx.context)
    qc = result.final_output
    preview = str(message).strip()
    if len(preview) > 400:
        preview = preview[:397] + "..."

    if qc.is_valid:
        send_push(f"Good query (sports): {preview}")
    else:
        send_push(f"Bad query (sports): {preview}\nReason: {qc.reason}")

    return GuardrailFunctionOutput(
        output_info={"query_check": qc},
        tripwire_triggered=not qc.is_valid,
    )


class SportFact(BaseModel):
    fact: str = Field(description="A little fact about the sport.")
    source: str = Field(description="The source of the fact.")

sport_agent = Agent(
    name="Sport Agent",
    model=openrouter_client,
    instructions="You are a sports expert. You are given a topic and you need to tell a little fact about it.",
    output_type=SportFact,
    input_guardrails=[guardrail_query],
)


In [6]:
async def run_sport_query(topic: str):
    """Run the sport agent. Returns SportFact, or None if the input guardrail blocks."""
    try:
        result = await Runner.run(sport_agent, topic)
        return result.final_output
    except InputGuardrailTripwireTriggered as e:
        qc = None
        try:
            qc = e.guardrail_result.output.output_info.get("query_check")
        except Exception:
            pass
        reason = getattr(qc, "reason", None) if qc else None
        print("Guardrail blocked:", reason or "Query not suitable for sports research.")
        return None

fact = await run_sport_query("Tell me an interesting historical fact about manchester united.")
if fact:
    print(fact.fact)
    print("Source:", fact.source)


AttributeError: 'function' object has no attribute 'get_name'